In [15]:
import pandas as pd
import numpy as np
import math

In [12]:
warehouses = pd.DataFrame({
    'warehouse_id': ['W1','W2','W3','W4','W5','W6','W7','W8'],
    'city': ['Casablanca','Rabat','Fès','Marrakech',
             'Agadir','Tanger','Oujda','Laâyoune'],
    'latitude': [33.5731, 34.0209, 34.0331, 31.6295,
                 30.4278, 35.7595, 34.6814, 27.1418],
    'longitude': [-7.5898, -6.8416, -5.0003, -7.9811,
                  -9.5981, -5.8340, -1.9086, -13.1800],
    'capacity_tonnes': [12000, 8000, 7000, 9000,
                        6000, 7500, 5000, 4000],
    'monthly_cost_mad': [180000, 120000, 100000, 130000,
                         90000, 110000, 80000, 70000]
})

stores = pd.DataFrame({
    'store_id': [f'S{str(i).zfill(3)}' for i in range(1, 21)],
    'city': ['Casablanca','Rabat','Fès','Marrakech','Agadir',
             'Tanger','Oujda','Laâyoune','Meknès','Béni Mellal',
             'Kénitra','Tétouan','Safi','El Jadida','Nador',
             'Settat','Khouribga','Essaouira','Ouarzazate','Dakhla'],
    'region': ['Centre','Centre','Nord','Sud','Sud',
               'Nord','Est','Sud','Centre','Centre',
               'Nord','Nord','Ouest','Ouest','Est',
               'Centre','Centre','Ouest','Sud','Sud'],
    'latitude': [33.5731, 34.0209, 34.0331, 31.6295, 30.4278,
                 35.7595, 34.6814, 27.1418, 33.8935, 32.3372,
                 34.2610, 35.5785, 32.2994, 33.2316, 35.1740,
                 33.0014, 32.8811, 31.5125, 30.9335, 23.7136],
    'longitude': [-7.5898, -6.8416, -5.0003, -7.9811, -9.5981,
                  -5.8340, -1.9086, -13.1800, -5.5473, -6.3498,
                  -6.5802, -5.3684, -9.2372, -8.5007, -2.9287,
                  -7.6166, -6.9063, -9.7749, -6.9370, -15.9355],
    'monthly_demand_tonnes': [450, 320, 280, 310, 190,
                               220, 150, 80, 200, 170,
                               180, 160, 140, 150, 120,
                               130, 110, 90, 85, 60]
})
warehouses.set_index('warehouse_id', inplace=True)
stores.set_index('store_id', inplace=True)


In [8]:
warehouses.to_csv('../data/raw/warehouses.csv', index=False)
stores.to_csv('../data/raw/stores.csv', index=False)

In [24]:
from geopy import distance

distances = []
def get_cord(row):
    return (row['latitude'],row['longitude'])
for _,wh in warehouses.iterrows():
    for _,st in stores.iterrows():
        dist = distance.geodesic(get_cord(wh),get_cord(st)).km
        distances.append({
            'warehouse_id': wh.name,
            'store_id': st.name,
            'distance_km': dist
        })

distances_df = pd.DataFrame(distances)
distances_df['transport_cost_mad'] = distances_df['distance_km']* 15
distances_df['delivery_days'] = np.ceil(distances_df['distance_km'] / 500).astype(int)
distances_df['is_feasible'] = distances_df['delivery_days'] <=2
distances_df.to_csv('../data/raw/distance_matrix.csv', index=False)
        